In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table

from dash import callback_context

from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo - DONE!
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    
    html.Center(
        html.A(
            href="https://www.snhu.edu",   
            target="_blank",                      
            children=html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={
                    'height': '120px',
                    'padding': '10px',
                    'cursor': 'pointer'
                }
            )
        )
    ),
    
    html.Center(html.B(html.H1('CS-340 Dashboard – Mark Vincent Francisco'))),
    html.Hr(),
    html.Div(
    
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc. - DONE!
        className='buttonRow',
            style={'display': 'flex'},
            children=[
                dcc.RadioItems(
        id='animal-radio',
        options=[
            {'label': 'Water Rescue', 'value': 'waterRescue'},
            {'label': 'Mountain Rescue', 'value': 'mountainRescue'},
            {'label': 'Disaster Rescue', 'value': 'disasterRescue'},
            {'label': 'Reset', 'value': 'reset'}
        ],
        value=None,
        inline=True
    )
]
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
                        editable=False,
                        filter_action="native",
                        sort_action="native",
                        sort_mode="multi",
                        column_selectable=False,
                        row_selectable="single",
                        row_deletable=False,
                        selected_columns=[],
                        selected_rows=[0],
                        page_action="native",
                        page_current=0,
                        page_size=10
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(Output('datatable-id','data'), 
        Output('datatable-id', 'columns'),
        Input('animal-radio', 'value')
             )
def update_dashboard(selected_value):
## FIX ME Add code to filter interactive data table with MongoDB queries

    age_filter = {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
    age_filter1 = {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}

    if selected_value == 'waterRescue':
        query = {"sex_upon_outcome": "Intact Female", 
                 "breed": {"$in": ['Newfoundland', 'Newfoundland Mix', 'Labrador Retriever Mix', 'Chesa Bay Retr Mix', 'Labrador Retriever/Chesa Bay Retr']}, **age_filter}
    elif selected_value == 'mountainRescue':
        query = {"sex_upon_outcome": "Intact Male", 
                 "breed": {"$in": ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']}, **age_filter}
    elif selected_value == 'disasterRescue':
        query = {"sex_upon_outcome": "Intact Male", 
                 "breed": {"$in": ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']}, **age_filter1}
    elif selected_value == 'reset':
        query = {} 
    else:
        query = {}

    records = db.read(query) or []
    dff = pd.DataFrame.from_records(records)
    
    if dff.empty:
        return [], []
    
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
#        
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in dff.columns]
    data=dff.to_dict('records')
#       
    return data,columns




# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if viewData is None or len(viewData) == 0:
        return [html.P("No data to display or Loading.")]

    dff = pd.DataFrame.from_records(viewData)

    if 'breed' not in dff.columns:
        return [html.P("Column 'breed' not found.")]
    
    breed_counts = (
        dff['breed']
        .value_counts()
        .nlargest(9)      # <-- displays only 9 breeds on the pie chart
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                names='breed',
                values='count',
                title='Preferred Animals'
            )
        )
    ]
    
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if not viewData:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    
    row = 0
    if index and 0 <= index[0] < len(dff):
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(port=8051) 

Dash app running on https://levellaura-bronzemiguel-3000.codio.io/proxy/8051/
